In [ ]:
import os
import time
from logging import INFO

from pathlib import Path
from pollen_worker.virtual_client import VirtualClient, gen_client_fn
from flwr.common import NDArrays, log
from flwr.server.strategy.aggregate import aggregate

# srun -w mauao -c 5 --partition=interactive --pty bash
# srun -w ngongotaha -c 5 --partition=interactive --pty bash
# poetry shell
# jupyter server --no-browser --port=8889
# ssh -L 8889:localhost:8889 ls985@mauao
# ssh -L 8889:localhost:8889 ls985@ngongotaha
HOME_DIR = Path(f"{os.getenv('HOME', '')}")

In [ ]:
task = "shakespeare_memory"
client_fn = gen_client_fn(task)
n_clients = 100
clients: list[VirtualClient] = [client_fn(0) for _ in range(n_clients)]
fake_fit_res: list[tuple[NDArrays, int]] = [(client.get_parameters({}), 1) for client in clients]
start_time = time.time()
aggregate(fake_fit_res)
log(INFO, f"Time elapsed: {time.time() - start_time}")

In [ ]:
results_dict: dict[int, dict[str, float]] = {}
# for n_clients in [10, 100, 1000, 10000]:
for n_clients in [10]:
    results_dict[n_clients]: dict[str, float] = {}
    for task in ["openimage", "google_speech", "reddit", "shakespeare_memory"]:
        client_fn = gen_client_fn(task)
        clients: list[VirtualClient] = [client_fn(0) for _ in range(n_clients)]
        fake_fit_res: list[tuple[NDArrays, int]] = [(client.get_parameters({}), 1) for client in clients]
        start_time = time.time()
        aggregate(fake_fit_res)
        elapsed_time = time.time() - start_time
        results_dict[n_clients][task] = elapsed_time
        log(INFO, f"Aggregating {n_clients} {task} clients, time elapsed: {elapsed_time}")
with open(HOME_DIR/"projects"/"pollen_worker"/"notebooks"/"aggregation_scaling_results.txt", "w") as f:
    f.write(str(results_dict))